In [1]:
import os
import json
from openai import OpenAI


In [2]:
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY")

if OPENAI_API_KEY:
    os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

In [3]:
client = OpenAI()

In [4]:
response = client.chat.completions.create(
    model="gpt-4.1-nano-2025-04-14",
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Who won the world series in 2020?"},
    ]
)

In [5]:
response.choices[0].message.content

'The Los Angeles Dodgers won the World Series in 2020.'

In [6]:
with open(file="data/customer_data.jsonl", mode="r", encoding="utf-8") as file:
  data = [json.loads(line) for line in file]

In [8]:
# data

In [9]:
data[0]

{'messages': [{'role': 'system',
   'content': 'You are a customer support assistant for a smartphone company. You are friendly, concise, and provide only factual answers related to smartphones.'},
  {'role': 'user',
   'content': 'How long does the warranty last on a new smartphone?'},
  {'role': 'assistant',
   'content': 'Most smartphones include a one-year limited warranty that covers manufacturing defects. Exact details are available in the official documentation.'}]}

In [10]:
data[1]

{'messages': [{'role': 'system',
   'content': 'You are a customer support assistant for a smartphone company. You are friendly, concise, and provide only factual answers related to smartphones.'},
  {'role': 'user',
   'content': 'Can I use the same apps on Android and iPhone?'},
  {'role': 'assistant',
   'content': 'Not all apps are available on both platforms. Some apps are exclusive to either Android or iOS, depending on the app store.'}]}

In [11]:
print("Num examples:", len(data))

Num examples: 10


In [12]:
for message in data[1]["messages"]:
  print(message)

{'role': 'system', 'content': 'You are a customer support assistant for a smartphone company. You are friendly, concise, and provide only factual answers related to smartphones.'}
{'role': 'user', 'content': 'Can I use the same apps on Android and iPhone?'}
{'role': 'assistant', 'content': 'Not all apps are available on both platforms. Some apps are exclusive to either Android or iOS, depending on the app store.'}


## Finetuning

In [13]:
client.files.create(
    file=open(file="data/customer_data.jsonl", mode="rb"),
    purpose="fine-tune"
)

FileObject(id='file-QQkcH3kXPKTHjULKRJiZzd', bytes=4248, created_at=1786372596, filename='customer_data.jsonl', object='file', purpose='fine-tune', status='processed', expires_at=None, status_details=None)

In [14]:
client.files.list()

SyncCursorPage[FileObject](data=[FileObject(id='file-QQkcH3kXPKTHjULKRJiZzd', bytes=4248, created_at=1786372596, filename='customer_data.jsonl', object='file', purpose='fine-tune', status='processed', expires_at=None, status_details=None), FileObject(id='file-NNchyyggKuMMUBJriFtYjB', bytes=4248, created_at=1786372473, filename='customer_data.jsonl', object='file', purpose='fine-tune', status='processed', expires_at=None, status_details=None)], has_more=False, object='list', first_id='file-QQkcH3kXPKTHjULKRJiZzd', last_id='file-NNchyyggKuMMUBJriFtYjB')

In [15]:
for file in client.files.list():
  print(file.id)
  print(100*"=")
  print(file.purpose)
  print(100*"=")
  print(file)


file-QQkcH3kXPKTHjULKRJiZzd
fine-tune
FileObject(id='file-QQkcH3kXPKTHjULKRJiZzd', bytes=4248, created_at=1786372596, filename='customer_data.jsonl', object='file', purpose='fine-tune', status='processed', expires_at=None, status_details=None)
file-NNchyyggKuMMUBJriFtYjB
fine-tune
FileObject(id='file-NNchyyggKuMMUBJriFtYjB', bytes=4248, created_at=1786372473, filename='customer_data.jsonl', object='file', purpose='fine-tune', status='processed', expires_at=None, status_details=None)


In [16]:
client.fine_tuning.jobs.create(
  training_file="file-NNchyyggKuMMUBJriFtYjB",
  model="gpt-4o-2024-08-06",
  suffix="openai-finetuning-testing",
  method={
    "type": "supervised",
    "supervised": {
      "hyperparameters": {
        "learning_rate_multiplier": 1.0,
        "n_epochs": 3
      }
    }
  }
)


PermissionDeniedError: Error code: 403 - {'error': {'message': 'OpenAI is winding down the fine-tuning platform and your organization is no longer able to create new fine-tuning training jobs. Learn more https://developers.openai.com/api/docs/deprecations#update-to-openais-self-serve-fine-tuning', 'type': 'invalid_request_error', 'param': None, 'code': 'training_not_available'}}

In [ ]:
https://developers.openai.com/api/docs/guides/model-optimization#fine-tune-a-model

In [ ]:
for job in client.fine_tuning.jobs.list():
  print(job.fine_tuned_model)

ft:gpt-4o-2024-08-06:personal:genaibootcamp-live-class:DtEzD4Nc
None
None
ft:gpt-4o-2024-08-06:personal:second-finetune-model:Cybog6kA
None


In [ ]:
client.chat.completions.create(
    model="ft:gpt-4o-2024-08-06:personal:genaibootcamp-live-class:DtEzD4Nc",
    messages=[
        {"role": "system", "content": "You are a customer support agent for a smartphone company whose primary goal is to help users with issues they are experiencing with their smartphones. You are friendly and concise. You only provide factual answers to queries, and do not provide answers that are not related to smartphones."},
        {
            "role": "user",
            "content": "What warranty does a smartphone come with?"
        }
    ]
)

ChatCompletion(id='chatcmpl-DtFH0NrrMHaqqGWJOFw1NXbWxafeM', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='Most smartphones come with a one-year limited warranty covering manufacturing defects. Check your specific model’s warranty terms for details.', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None))], created=1782059122, model='ft:gpt-4o-2024-08-06:personal:genaibootcamp-live-class:DtEzD4Nc', object='chat.completion', moderation=None, service_tier='default', system_fingerprint='fp_09d511fdef', usage=CompletionUsage(completion_tokens=23, prompt_tokens=71, total_tokens=94, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0)))

In [ ]:
answer = client.chat.completions.create(
    model="ft:gpt-4o-2024-08-06:personal:genaibootcamp-live-class:DtEzD4Nc",
    messages=[
        {"role": "system", "content": "You are a customer support agent for a smartphone company whose primary goal is to help users with issues they are experiencing with their smartphones. You are friendly and concise. You only provide factual answers to queries, and do not provide answers that are not related to smartphones."},
        {
            "role": "user",
            "content": "Can I use the same apps on Android and iPhone?"
        }
    ]
)

In [42]:
print(answer.choices[0].message.content)

Apps need to be downloaded separately from Google Play Store for Android and the App Store for iPhone. Compatibility depends on the developer.
